In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%%bash
# 1. Create local workspace
mkdir -p /content/KITTI_Project

# 2. Copy code and configs from Drive
cp -r /content/drive/MyDrive/KITTI_Project/src /content/KITTI_Project/
cp -r /content/drive/MyDrive/KITTI_Project/configs /content/KITTI_Project/
cp /content/drive/MyDrive/KITTI_Project/requirements.txt /content/KITTI_Project/

# 3. Copy and unzip data locally
mkdir -p /content/KITTI_Project/data
cp /content/drive/MyDrive/KITTI_Project/data/data.zip /content/KITTI_Project/data/
cp /content/drive/MyDrive/KITTI_Project/data/calib.zip /content/KITTI_Project/data/

unzip -q /content/KITTI_Project/data/data.zip -d /content/KITTI_Project/data/
unzip -q /content/KITTI_Project/data/calib.zip -d /content/KITTI_Project/data/

# 4. Flatten the nested KITTI folders (Pulling files out of subfolders)
mv /content/KITTI_Project/data/2011_09_26/* /content/KITTI_Project/data/ 2>/dev/null || true
mv /content/KITTI_Project/data/2011_09_26_drive_0064_sync/* /content/KITTI_Project/data/ 2>/dev/null || true

# 5. Final structure check
echo "--- Final Data Directory Structure ---"
ls -F /content/KITTI_Project/data/

# 6. Create output directory for experiment 0064
mkdir -p /content/KITTI_Project/output/exp_0064

--- Final Data Directory Structure ---
2011_09_26/
2011_09_26_drive_0064_sync/
calib_cam_to_cam.txt
calib_imu_to_velo.txt
calib_velo_to_cam.txt
calib.zip
data.zip
image_00/
image_01/
image_02/
image_03/
oxts/
velodyne_points/


In [3]:
%%bash
# Move LiDAR files out of the nested data/ subfolder
mv /content/KITTI_Project/data/velodyne_points/data/* /content/KITTI_Project/data/velodyne_points/ 2>/dev/null || true

# Move image files out of the nested data/ subfolder
mv /content/KITTI_Project/data/image_02/data/* /content/KITTI_Project/data/image_02/ 2>/dev/null || true

# Move oxts files out of the nested data/ subfolder
mv /content/KITTI_Project/data/oxts/data/* /content/KITTI_Project/data/oxts/ 2>/dev/null || true

# Verify that the actual files are now visible (you should see .bin or .png files)
echo "--- Top 3 LiDAR Files ---"
ls /content/KITTI_Project/data/velodyne_points/ | head -n 3

echo "--- Top 3 Image Files ---"
ls /content/KITTI_Project/data/image_02/ | head -n 3

--- Top 3 LiDAR Files ---
0000000000.bin
0000000001.bin
0000000002.bin
--- Top 3 Image Files ---
0000000000.png
0000000001.png
0000000002.png


In [4]:
%cd /content/KITTI_Project
!pip install -q -r requirements.txt

/content/KITTI_Project
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 116.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# !pip install transformers -q

import os
import glob
import cv2
import torch
import numpy as np
from PIL import Image
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

# 1. Load the pre-trained Autonomous Driving AI
print("Loading SegFormer AI...")
processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b0-finetuned-cityscapes-512-1024")
model = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b0-finetuned-cityscapes-512-1024").to("cuda")

# Create/ensure the masks directory exists
os.makedirs('data/masks', exist_ok=True)
image_paths = sorted(glob.glob('data/image_02/*.png'))[:50]

print(f"Generating intelligent masks for {len(image_paths)} frames...")

# In the Cityscapes dataset, the ID for "sky" is exactly 10.
SKY_CLASS_ID = 10

with torch.no_grad():
    for img_path in image_paths:
        # Load image with PIL for the HuggingFace processor
        image = Image.open(img_path).convert("RGB")
        original_size = image.size[::-1] # (height, width)
        
        # 2. Run the AI Prediction
        inputs = processor(images=image, return_tensors="pt").to("cuda")
        outputs = model(**inputs)
        logits = outputs.logits
        
        # 3. Resize prediction back to original KITTI image size
        upsampled_logits = torch.nn.functional.interpolate(
            logits,
            size=original_size,
            mode="bilinear",
            align_corners=False
        )
        
        # Get the predicted class for every single pixel
        predicted_classes = upsampled_logits.argmax(dim=1).squeeze().cpu().numpy()
        
        # 4. Build the Mask (White = Keep, Black = Sky)
        # Create an all-white mask (255)
        mask = np.ones_like(predicted_classes, dtype=np.uint8) * 255
        
        # Wherever the AI saw "Sky" (class 10), paint it black (0)
        mask[predicted_classes == SKY_CLASS_ID] = 0
        
        # Save it
        name = os.path.basename(img_path)
        cv2.imwrite(f'data/masks/{name}', mask)

print("Masks successfully generated!")

In [ ]:
import cv2
import glob
import os
import numpy as np

mask_paths = sorted(glob.glob('data/masks/*.png'))
kernel = np.ones((3, 3), np.uint8) 

print("Eroding masks to kill sky-bleeding...")
for path in mask_paths:
    mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    
    # Shave 2 pixels off the edges of all objects
    eroded_mask = cv2.erode(mask, kernel, iterations=2)
    
    cv2.imwrite(path, eroded_mask)

print("Done! Horizon edges are now protected.")

In [5]:
import yaml
import torch
import numpy as np
import os

from src.data.kitti_dataset import KittiDataset
from src.models.gaussian_model import GaussianModel
from src.models.loss_functions import combined_loss
from src.models.trainer import SplatTrainer
from src.models.densifier import Densifier

# Load Configuration for experiment 0064
with open("configs/residential_0064.yaml", "r") as f:
    config = yaml.safe_load(f)

print(f"Loaded configuration for experiment: {config['experiment']['name']}")

Loaded configuration for experiment: residential_0064


In [ ]:
# 1. Take a continuous x-frame slice (e.g., 2 seconds of driving)  
dataset = KittiDataset(config)
downsample_rate = config['data'].get('downsample_rate', 5)

# 2. Stitch ONLY the LiDAR for those exact 15 frames
all_points = []
for i in range(len(dataset.w2c_matrices)): 
    lidar_path = os.path.join(config['data']['lidar_dir'], f"{i:010d}.bin")
    if not os.path.exists(lidar_path): continue
    
    scan = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 4)[:, :3]
    scan = scan[::downsample_rate] # Keep the downsampling so it stays small and fast!
    
    # --- NEW MATH ---
    # 1. Convert LiDAR (X-forward) to Camera Space (Z-forward)
    scan_hom = np.hstack((scan, np.ones((scan.shape[0], 1))))
    scan_cam = (dataset.Tr_v2c_rect @ scan_hom.T).T[:, :3]
    
    # 2. Convert Camera Space to World Space
    w2c = dataset.w2c_matrices[i]
    c2w = np.linalg.inv(w2c)
    scan_world = (scan_cam @ c2w[:3, :3].T) + c2w[:3, 3]
    # ----------------
    
    all_points.append(scan_world)

pcd_combined = np.concatenate(all_points, axis=0)
print(f"Starting with {len(pcd_combined)} points (much better!)")

# 3. Initialize Model
model = GaussianModel(sh_degree=config['model']['sh_degree']) 
model.create_from_pcd(pcd_xyz=pcd_combined, device="cuda")

# 4. Initialize the Densifier
densifier = Densifier(
    model=model,
    grad_threshold=config['densification']['grad_threshold'],
    opacity_threshold=config['densification']['opacity_threshold'],
    extent=config['densification']['spatial_extent']
)

# 5. Initialize the Trainer
trainer = SplatTrainer(
    gaussian_model=model,
    dataset=dataset,
    config=config,
    densifier=densifier,
    iterations=config['training']['iterations'],
    device=config['experiment']['device']
)
print("Pipeline fully initialized and ready for training.")

Successfully loaded KITTI dataset indexing 570 frames.
Initialized 124244 Gaussians from point cloud.
Pipeline fully initialized and ready for training.


In [7]:
!pip install ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 17.4 MB/s eta 0:00:00


In [8]:
# Start the optimization loop
trainer.train()

Training Splats:   0%|          | 0/30000 [00:00<?, ?it/s]

Output()

Training Splats:   0%|          | 0/30000 [01:09<?, ?it/s]


RuntimeError: Boolean value of Tensor with more than one value is ambiguous

In [ ]:
# 1. Define output paths
local_out_path = os.path.join(config['experiment']['workspace'], "point_cloud.ply")
drive_out_path = f"/content/drive/MyDrive/KITTI_Project/output/exp_0064/point_cloud.ply"

# 2. Save the PLY file locally
model.save_ply(local_out_path)

# 3. Copy back to Google Drive
import shutil
shutil.copy2(local_out_path, drive_out_path)
print(f"Training complete! Point cloud successfully backed up to Google Drive at: {drive_out_path}")

In [ ]:
import numpy.lib.recfunctions as rfn
from plyfile import PlyData, PlyElement
import shutil

def export_for_radiancekit(input_ply, output_ply):
    print(f"Scanning {input_ply} for corruption...")
    
    # 1. Load the original PLY
    plydata = PlyData.read(input_ply)
    vertex_data = plydata.elements[0].data
    
    # 2. Convert structured array to a flat matrix to check for math errors
    unstructured = rfn.structured_to_unstructured(vertex_data)
    
    # 3. Find any rows that contain NaNs or Infinity
    valid_mask = np.isfinite(unstructured).all(axis=1)
    clean_data = vertex_data[valid_mask]
    
    dropped = len(vertex_data) - len(clean_data)
    if dropped > 0:
        print(f"⚠️ Removed {dropped} corrupted Gaussians (NaN/Inf detected)!")
    else:
        print("✅ No corruption found. Data is clean.")
        
    # 4. Save specifically as Binary Little Endian (RadianceKit's requirement)
    el = PlyElement.describe(clean_data, 'vertex')
    PlyData([el], text=False, byte_order='<').write(output_ply)
    print(f"Successfully saved RadianceKit file to {output_ply}")

# Run the sanitization on the file you just saved
rk_local_path = local_out_path.replace(".ply", "_radiancekit.ply")
rk_drive_path = drive_out_path.replace(".ply", "_radiancekit.ply")

export_for_radiancekit(local_out_path, rk_local_path)

# Back it up to Google Drive
shutil.copy2(rk_local_path, rk_drive_path)
print(f"RadianceKit PLY backed up to Drive!")